# Retweet Interaction Graph (Directed)

Nodes are users. A directed edge from user A to user B exists if A retweeted B, weighted by retweet frequency.

This notebook extracts retweet interactions directly from `tweets.csv` using the `retweetedTweet` metadata field.
Update `INPUT_PATH` below to match your Kaggle dataset path.


In [ ]:
# Optional: install if needed
# !pip -q install -U pandas networkx tqdm

import ast
import json
import os
import math
from collections import Counter

import pandas as pd
import networkx as nx

try:
    from tqdm import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

# ==== PARAMETERS ====
INPUT_PATH = '/kaggle/input/farmers-protest-data/tweets.csv'
OUTDIR = '/kaggle/working/retweet_graph'
CHUNKSIZE = 200000
MAX_ROWS = 0  # 0 means all rows; set for quick tests, e.g. 200000


In [ ]:
def parse_retweeted(obj):
    if obj is None:
        return None
    if isinstance(obj, float) and math.isnan(obj):
        return None
    if isinstance(obj, dict):
        return obj
    s = str(obj).strip()
    if not s or s.lower() in {'nan', 'none', 'null'}:
        return None
    # Try JSON first
    try:
        return json.loads(s)
    except Exception:
        pass
    # Fallback: Python literal (often single-quoted dict)
    try:
        return ast.literal_eval(s)
    except Exception:
        return None

def extract_original_user_id(retweeted_obj):
    if retweeted_obj is None:
        return None
    if isinstance(retweeted_obj, list) and retweeted_obj:
        retweeted_obj = retweeted_obj[0]
    if not isinstance(retweeted_obj, dict):
        return None

    # Common patterns
    if 'user' in retweeted_obj and isinstance(retweeted_obj['user'], dict):
        user_obj = retweeted_obj['user']
        for key in ('id', 'id_str', 'userId', 'userid'):
            if key in user_obj and user_obj[key]:
                return str(user_obj[key])

    for key in ('userId', 'userid', 'authorId', 'author_id'):
        if key in retweeted_obj and retweeted_obj[key]:
            return str(retweeted_obj[key])

    return None


In [ ]:
# Diagnostic: inspect retweetedTweet structure
sample = []
read_kwargs_diag = {
    'usecols': ['retweetedTweet'],
    'chunksize': 20000,
    'dtype': {'retweetedTweet': 'string'},
    'on_bad_lines': 'skip',
    'na_filter': False,
}
if INPUT_PATH.endswith('.gz'):
    read_kwargs_diag['compression'] = 'gzip'

for chunk in pd.read_csv(INPUT_PATH, **read_kwargs_diag):
    for rt in chunk['retweetedTweet']:
        if not rt:
            continue
        retweeted_obj = parse_retweeted(rt)
        if retweeted_obj is None:
            continue
        sample.append((rt, retweeted_obj, extract_original_user_id(retweeted_obj)))
        if len(sample) >= 5:
            break
    if len(sample) >= 5:
        break

for i, (raw, obj, uid) in enumerate(sample, 1):
    print(f'--- sample {i} ---')
    raw_str = str(raw)
    print('raw:', raw_str[:500])
    if isinstance(obj, list) and obj:
        keys = list(obj[0].keys()) if isinstance(obj[0], dict) else None
    elif isinstance(obj, dict):
        keys = list(obj.keys())
    else:
        keys = None
    print('parsed type:', type(obj))
    print('parsed keys:', keys)
    print('extracted userId:', uid)


In [ ]:
edge_counts = Counter()
retweet_rows = 0
processed = 0

read_kwargs = {
    'usecols': ['userId', 'retweetedTweet'],
    'chunksize': CHUNKSIZE,
    'dtype': {'userId': 'string', 'retweetedTweet': 'string'},
    'on_bad_lines': 'skip',
    'na_filter': False,
}

if INPUT_PATH.endswith('.gz'):
    read_kwargs['compression'] = 'gzip'

for chunk in tqdm(pd.read_csv(INPUT_PATH, **read_kwargs), desc='Scanning tweets'):
    for uid, rt in zip(chunk['userId'], chunk['retweetedTweet']):
        processed += 1
        if MAX_ROWS and processed > MAX_ROWS:
            break
        if not rt:
            continue
        retweeted_obj = parse_retweeted(rt)
        original_id = extract_original_user_id(retweeted_obj)
        if not original_id:
            continue
        if not uid:
            continue
        edge_counts[(str(uid), str(original_id))] += 1
        retweet_rows += 1
    if MAX_ROWS and processed > MAX_ROWS:
        break

print('Processed rows:', processed)
print('Retweet rows with extracted original user:', retweet_rows)
print('Unique edges:', len(edge_counts))


In [ ]:
# Build directed graph and export
G = nx.DiGraph()
for (u, v), w in edge_counts.items():
    G.add_edge(u, v, weight=int(w))

# Compute basic node stats
in_weight = {n: 0 for n in G.nodes()}
out_weight = {n: 0 for n in G.nodes()}
for u, v, d in G.edges(data=True):
    w = d.get('weight', 0)
    out_weight[u] += w
    in_weight[v] += w

os.makedirs(OUTDIR, exist_ok=True)
edges_path = os.path.join(OUTDIR, 'retweet_edges.csv')
nodes_path = os.path.join(OUTDIR, 'retweet_nodes.csv')
gexf_path = os.path.join(OUTDIR, 'retweet_graph.gexf')

edges_df = pd.DataFrame([
    {'source': u, 'target': v, 'weight': d['weight']}
    for u, v, d in G.edges(data=True)
])
edges_df.to_csv(edges_path, index=False)

nodes_df = pd.DataFrame({
    'userId': list(G.nodes()),
    'in_weight': [in_weight[n] for n in G.nodes()],
    'out_weight': [out_weight[n] for n in G.nodes()],
    'in_degree': [G.in_degree(n) for n in G.nodes()],
    'out_degree': [G.out_degree(n) for n in G.nodes()],
})
nodes_df.to_csv(nodes_path, index=False)

nx.write_gexf(G, gexf_path)

stats = {
    'nodes': G.number_of_nodes(),
    'edges': G.number_of_edges(),
    'retweet_rows_used': retweet_rows,
}
with open(os.path.join(OUTDIR, 'stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print('Done:', stats)
